In [1]:
from pathlib import Path
import sys
import pandas as pd
from tqdm import tqdm

sys.path.insert(0, str(Path('..').resolve()))

from lib.ai import vlmrun_score_aesthetic, gemini_score_aesthetic, gpt5_score_aesthetic

print("Multi-model aesthetic scoring setup complete")

Multi-model aesthetic scoring setup complete


In [2]:
# Load all spec renders
specs_dir = Path('../datasets/specs/')
spec_paths = sorted(list(specs_dir.glob('*/render.png')))

# Create dataframe with template IDs and render paths
specs_df = pd.DataFrame({
    'template_id': [p.parent.name for p in spec_paths],
    'render_path': spec_paths
})

print(f"Found {len(specs_df)} design spec renders")
specs_df.head()

Found 100 design spec renders


,template_id,render_path
0,1067w-IQprojiENUA,../datasets/specs/1067w-IQprojiENUA/render.png
1,1131w-02GXIIidf_4,../datasets/specs/1131w-02GXIIidf_4/render.png
2,1131w-120Kpwxx3eY,../datasets/specs/1131w-120Kpwxx3eY/render.png
3,1131w-50t2MzJLmsM,../datasets/specs/1131w-50t2MzJLmsM/render.png
4,1131w-9cN5biKALLQ,../datasets/specs/1131w-9cN5biKALLQ/render.png


In [ ]:
# Score each render with VLM Run PRO model (parallelized)
from concurrent.futures import ThreadPoolExecutor, as_completed

def score_single_render_vlmrun_pro(row):
    """Score a single render with VLM Run PRO and return (template_id, score)"""
    template_id = row['template_id']
    render_path = row['render_path']
    
    try:
        score = vlmrun_score_aesthetic(render_path, model="vlmrun-orion-1:pro")
        return template_id, score
    except Exception as e:
        print(f"VLM Run PRO error for {template_id}: {e}")
        return template_id, None

# Parallelize scoring with ThreadPoolExecutor
max_workers = 10
pro_scores_dict = {}

print(f"Scoring {len(specs_df)} renders with VLM Run PRO using {max_workers} workers...")

with ThreadPoolExecutor(max_workers=max_workers) as executor:
    futures = {executor.submit(score_single_render_vlmrun_pro, row): idx for idx, row in specs_df.iterrows()}
    
    for future in tqdm(as_completed(futures), total=len(futures), desc="Scoring with VLM Run PRO"):
        template_id, score = future.result()
        pro_scores_dict[template_id] = score

# Add PRO scores to dataframe
specs_df['vlmrun_pro_score'] = specs_df['template_id'].map(pro_scores_dict)

print(f"\nCompleted PRO scoring for {len(specs_df)} renders")
print("\nVLM Run PRO Score Summary:")
print(specs_df['vlmrun_pro_score'].describe())

specs_df.head()

In [ ]:
# Save results to CSV
output_path = Path('spec_aesthetic_scores_vlmrun_pro.csv')
specs_df.to_csv(output_path, index=False)
print(f"Saved results to {output_path}")

# Show summary statistics
print(f"\nVLM Run PRO Score Summary:")
print(specs_df['vlmrun_pro_score'].describe())

specs_df

In [ ]:
# Visualize VLM Run PRO scores
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Score distribution histogram
ax1 = axes[0]
scores = specs_df['vlmrun_pro_score'].dropna()
ax1.hist(scores, bins=20, color='#2E7D32', alpha=0.7, edgecolor='black')
ax1.axvline(scores.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {scores.mean():.1f}')
ax1.axvline(scores.median(), color='orange', linestyle='--', linewidth=2, label=f'Median: {scores.median():.1f}')
ax1.set_xlabel('VLM Run PRO Score')
ax1.set_ylabel('Frequency')
ax1.set_title('VLM Run PRO Score Distribution')
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

# 2. Top and bottom designs
ax2 = axes[1]
sorted_df = specs_df.sort_values('vlmrun_pro_score', ascending=False)
top_10 = sorted_df.head(10)
bottom_10 = sorted_df.tail(10)

y_pos = np.arange(10)
ax2.barh(y_pos, top_10['vlmrun_pro_score'].values, color='#4CAF50', alpha=0.6, label='Top 10')
ax2.barh(y_pos + 11, bottom_10['vlmrun_pro_score'].values, color='#F44336', alpha=0.6, label='Bottom 10')
ax2.set_yticks(list(y_pos) + list(y_pos + 11))
ax2.set_yticklabels(list(top_10['template_id'].values) + list(bottom_10['template_id'].values), fontsize=8)
ax2.set_xlabel('VLM Run PRO Score')
ax2.set_title('Top 10 and Bottom 10 Designs')
ax2.legend()
ax2.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('vlmrun_pro_scores.png', dpi=150, bbox_inches='tight')
plt.show()

print("Saved visualization to vlmrun_pro_scores.png")